In [1]:
from experiments.dj.likelihood_tables import LikelihoodConfig
from experiments.dj.result_tables import (
    AdaptPriorResult,
    FlowPriorResult,
    LikelihoodResult,
)
from experiments.dj.prior_tables import FlowPriorConfig
from experiments.dj.trainer_tables import FPTrainerConfig, LLTrainerConfig
from experiments.dj.dataloader_tables import DataLoaderConfig
from experiments.dj.dj_helpers import fetch_best_model_results
import gensn.distributions as G
import torch
from task_transfer.ml_lib.data_loading import build_dataloaders
from task_transfer.evaluation.evaluate_generative_model import (
    evaluate_flow_prior,
    compute_logl,
    adapt_prior_eval_criterion,
    compute_joint_logl,
    logl_mc_marginal_eval,
)


from experiments.dj.posterior_tables import SBVGPConfig

from experiments.dj.sysident_tables import SIConfig
from experiments.dj.result_tables import (
    SBVGPResult,
    SIResult,
    FlowPriorResult,
    AdaptPriorResult,
)
from experiments.dj.dataloader_tables import DataLoaderConfig

from task_transfer.utils.insilico_stimuli import generate_gabors
from task_transfer.ml_lib.data_loading import build_dataloaders

from task_transfer.evaluation.evaluate_generative_model import compute_logl
from task_transfer.evaluation.evaluate_generative_model import (
    compute_logl_marginal,
    compute_logl_data_marginal,
)
import torch
import matplotlib.pyplot as plt
import seaborn as sns


import numpy as np
from task_transfer.utils.math_utils import cos2_von_mises

from task_transfer.utils.model_utils import build_haefner_model

torch.manual_seed(42)

import experiments.orientation_discrimination.haefner_model.configs as cfg

[2024-07-23 12:06:28,746][INFO]: Connecting sshrinivasan@134.76.19.44:3306
[2024-07-23 12:06:28,754][INFO]: Connected sshrinivasan@134.76.19.44:3306
/usr/local/lib/python3.8/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: libtorch_cuda_cu.so: cannot open shared object file: No such file or directory
  warn(f"Failed to load image Python extension: {e}")


In [2]:
dataloader_args = {
    "data_fname": "/src/project/data/synthetic/haefner_2afc/flat_haefner_100k_dataset.pkl",
    "train_prop": 0.7,
    "val_prop": 0.2,
}
train_loader, val_loader, test_loader = build_dataloaders(
    data_fname=dataloader_args["data_fname"],
    train_prop=dataloader_args["train_prop"],
    val_prop=dataloader_args["val_prop"],
    batch_size=128,
)

In [3]:
true_prior = G.IndependentExponential(rate=torch.ones(45))

In [4]:
download_path = "/tmp"
criterion = "val_ll_mean"
k = 1

direct_flow_restriction = f"dl_id = 'b8379e7d6998fc94a08a9a3742eec12d'"
prior_result_table = FlowPriorResult & direct_flow_restriction

direct_flow_dataloader_restrictions = (
    f"data_fname = '/src/project/data/synthetic/haefner_2afc/flat_haefner_dataset.pkl'"
)
dataloader_table = DataLoaderConfig & direct_flow_dataloader_restrictions

prior_config_proj_col = "fp_id"
best_val_prior_results = fetch_best_model_results(
    result_table=prior_result_table,
    config_table=FlowPriorConfig,
    data_loader_config_table=dataloader_table,
    trainer_config_table=FPTrainerConfig,
    config_proj_col=prior_config_proj_col,
    criterion=criterion,
    k=k,
    download_path=download_path,
)
direct_flow = torch.load(best_val_prior_results["model"], map_location="cpu")

# get likelihood model for marginal likelihood
likelihood_config_proj_col = "ll_id"
likelihood_restrictions = restriction = "dl_id = 'b8379e7d6998fc94a08a9a3742eec12d'"
likelihood_result_table = LikelihoodResult & likelihood_restrictions

best_val_likelihood_results = fetch_best_model_results(
    result_table=likelihood_result_table,
    config_table=LikelihoodConfig,
    data_loader_config_table=dataloader_table,
    trainer_config_table=LLTrainerConfig,
    config_proj_col=likelihood_config_proj_col,
    criterion=criterion,
    k=k,
    download_path=download_path,
)
likelihood = torch.load(best_val_likelihood_results["model"], map_location="cpu")

In [5]:
import experiments.orientation_discrimination.haefner_model.configs as data_cfg
from task_transfer.utils.model_utils import build_haefner_model
import torch
import torch.nn as nn
import gensn.distributions as G
cfg = data_cfg.flat_haefner_100k
true_model = build_haefner_model(
    p_c=cfg["p_c"],
    c1_psi=cfg["c1_psi"],
    c2_psi=cfg["c2_psi"],
    kappa=cfg["kappa"],
    g_phi=cfg["g_phi"],
    delta=cfg["delta"],
    lam=cfg["lam"],
    x_phi=cfg["x_phi"],
    obs_sigma=cfg["obs_sigma"],
    obs_h=cfg["obs_h"],
    obs_w=cfg["obs_w"],
)
class TrueLikelihood(nn.Module):
    def __init__(self, in_features, out_features, weight_init, bias_init, sigma):
        super().__init__()
        self.fn = nn.Linear(in_features, out_features)
        self.fn.weight.data = weight_init
        self.fn.bias.data = bias_init
        self.sigma = sigma

    def forward(self, x):
        return self.fn(x), self.sigma

weight = true_model.x_pfs.view(true_model.x_pfs.shape[0], -1).T
bias = torch.zeros(weight.shape[0])
sigma = torch.nn.Parameter(torch.tensor(0.1))
true_likelihood = TrueLikelihood(weight.shape[1], weight.shape[0], weight, bias, sigma)
true_conditional = G.IndependentNormal(_parameters=true_likelihood)
likelihood = true_conditional

In [6]:
weight.shape

torch.Size([144, 45])

In [7]:
pt_exp_restrictions = (
    "seed = 666 "
    "and orig_dl_id = 'b8379e7d6998fc94a08a9a3742eec12d' "
    "and dl_id = 'b8379e7d6998fc94a08a9a3742eec12d' "
)
pt_exp_adpt_prior = torch.load(
        (AdaptPriorResult & pt_exp_restrictions).fetch(
        download_path="/tmp",
        order_by="val_marginal_obs_ll_mean DESC",
        limit=1,
        as_dict=True,
    )[0]["model"],
    map_location="cpu",
)

rnd_exp_restrictions = (
    "seed = -666 "
    "and orig_dl_id = 'b8379e7d6998fc94a08a9a3742eec12d' "
    "and dl_id = 'b8379e7d6998fc94a08a9a3742eec12d' "
)
rnd_exp_adpt_prior = torch.load(
        (AdaptPriorResult & rnd_exp_restrictions).fetch(
        download_path="/tmp",
        order_by="val_marginal_obs_ll_mean DESC",
        limit=1,
        as_dict=True,
    )[0]["model"],
    map_location="cpu",
)

pt_flow_restrictions = (
    "(seed > 0 and seed != 666) "
    "and orig_dl_id = 'b8379e7d6998fc94a08a9a3742eec12d' "
    "and dl_id = 'b8379e7d6998fc94a08a9a3742eec12d' "
)
pt_adpt_flow = torch.load(
        (AdaptPriorResult & pt_flow_restrictions).fetch(
        download_path="/tmp",
        order_by="val_marginal_obs_ll_mean DESC",
        limit=1,
        as_dict=True,
    )[0]["model"],
    map_location="cpu",
)

rnd_flow_restrictions = (
    "(seed < 0 and seed != -666) "
    "and orig_dl_id = 'b8379e7d6998fc94a08a9a3742eec12d' "
    "and dl_id = 'b8379e7d6998fc94a08a9a3742eec12d' "
)
rnd_adpt_flow = torch.load(
        (AdaptPriorResult & rnd_flow_restrictions).fetch(
        download_path="/tmp",
        order_by="val_marginal_obs_ll_mean DESC",
        limit=1,
        as_dict=True,
    )[0]["model"],
    map_location="cpu",
)

# pt_adpt_flow_100k_restrictions = (
#     "(seed > 0 and seed != 666) "
#     "and orig_dl_id = 'b8379e7d6998fc94a08a9a3742eec12d' "
#     "and dl_id = 'd6b36dc9d4024882e4b7ccc597495a32' "
# )
# pt_adpt_flow_100k = torch.load(
#         (AdaptPriorResult & pt_adpt_flow_100k_restrictions).fetch(
#         download_path="/tmp",
#         order_by="val_marginal_obs_ll_mean DESC",
#         limit=1,
#         as_dict=True,
#     )[0]["model"],
#     map_location="cpu",
# )

# rnd_adpt_flow_100k_restrictions = (
#     "(seed < 0 and seed != -666) "
#     "and orig_dl_id = 'b8379e7d6998fc94a08a9a3742eec12d' "
#     "and dl_id = 'd6b36dc9d4024882e4b7ccc597495a32' "
# )
# rnd_adpt_flow_100k = torch.load(
#         (AdaptPriorResult & rnd_adpt_flow_100k_restrictions).fetch(
#         download_path="/tmp",
#         order_by="val_marginal_obs_ll_mean DESC",
#         limit=1,
#         as_dict=True,
#     )[0]["model"],
#     map_location="cpu",
# )

# pt_adpt_exp_100k_restrictions = (
#     "seed = 666 "
#     "and orig_dl_id = 'b8379e7d6998fc94a08a9a3742eec12d' "
#     "and dl_id = 'd6b36dc9d4024882e4b7ccc597495a32' "
# )
# pt_adpt_exp_100k = torch.load(
#         (AdaptPriorResult & pt_adpt_exp_100k_restrictions).fetch(
#         download_path="/tmp",
#         order_by="val_marginal_obs_ll_mean DESC",
#         limit=1,
#         as_dict=True,
#     )[0]["model"],
#     map_location="cpu",
# )

# rnd_adpt_exp_100k_restrictions = (
#     "seed = -666 "
#     "and orig_dl_id = 'b8379e7d6998fc94a08a9a3742eec12d' "
#     "and dl_id = 'd6b36dc9d4024882e4b7ccc597495a32' "
# )
# rnd_adpt_exp_100k = torch.load(
#         (AdaptPriorResult & rnd_adpt_exp_100k_restrictions).fetch(
#         download_path="/tmp",
#         order_by="val_marginal_obs_ll_mean DESC",
#         limit=1,
#         as_dict=True,
#     )[0]["model"],
#     map_location="cpu",
# )

In [33]:
from pathlib import Path


def visualize_marginal_flow(
    models,
    data_loader,
    device="cpu",
    density_support=(1e-3, 10),
    density_n_samples=1000,
    dims_to_plot=range(45),
    fig_dpi=300,
    linewidth=3,
    fontsize=10,
    plot_xlim=(0, 7),
    plot_ylim=(0, 1),
    data_color="darkorange",
    data_alpha=0.4,
    fig_save_dir=Path("/src/project/figures/learning/"),
    **catch_all,
):
    all_responses = []
    for batch in data_loader:
        responses, _ = batch
        all_responses.append(responses.to(device))
    all_responses = torch.cat(all_responses, dim=0)
    n_dims_to_plot = len(dims_to_plot)
    with torch.no_grad():
        n_dims_all = all_responses.shape[1]
        x = (
            torch.linspace(density_support[0], density_support[1], density_n_samples)
            .repeat(n_dims_all, 1)
            .T
        )
        fig, axs = plt.subplots(
            7,
            7,
            sharey=True,
            sharex=True,
            dpi=fig_dpi,
            tight_layout=True,
        )
        for idx, ax in zip(dims_to_plot, axs.ravel()):
            sns.histplot(
                all_responses[:, idx].detach().cpu(),
                ax=ax,
                stat="density",
                element="step",
                color=data_color,
                alpha=data_alpha,
                label="Data",
            )
        colors = sns.color_palette("tab10", n_colors=len(models))
        for model, color, label, linestyle in zip(
            models, colors, catch_all["labels"], catch_all["linestyles"]
        ):
            flow_density = model.factorized_log_prob(x.to(device)).exp().cpu().numpy()
            print(flow_density.shape)
            for idx, ax in zip(dims_to_plot, axs.ravel()):
                ax.plot(
                    x[:, idx].detach().cpu(),
                    flow_density[:, idx],
                    linewidth=linewidth,
                    color=color,
                    label=label,
                    linestyle=linestyle,
                )
                ax.set_xlim(*plot_xlim)
                ax.set_ylim(*plot_ylim)
                ax.set_xticks(plot_xlim)
                ax.set_xticklabels([f"{x:.1f}" for x in plot_xlim])
                # ax.axis("off")
                ax.tick_params(axis="both", which="both", labelsize=fontsize)
                sns.despine(ax=ax)
                ax.spines[["left", "top", "right"]].set_visible(False)
                ax.set_ylabel("$p(x)$", fontsize=fontsize)
                ax.set_xlabel("x", fontsize=fontsize)
            for ax in axs.ravel()[n_dims_to_plot:]:
                ax.axis("off")
        handles, labels = axs.flatten()[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="lower right", fontsize=fontsize)
        fig.savefig(
            fig_save_dir / catch_all["fig_name"],
            bbox_inches="tight",
            transparent=True,
        )
        # close the figure to avoid memory leak
        plt.close(fig)

In [54]:
device = "cpu"
density_support = (1e-3, 7)
xticks = (0, 7)
xticklabels = (0, 7)
plot_ylim = (0, 1)
visualize_marginal_flow(
    models=[true_prior, direct_flow, pt_adpt_flow.prior, rnd_adpt_flow.prior],
    data_loader=val_loader,
    device=device,
    density_support=density_support,
    density_n_samples=1000,
    dims_to_plot=range(45),
    fig_dpi=300,
    linewidth=2,
    fontsize=10,
    plot_xlim=xticks,
    plot_ylim=plot_ylim,
    data_color="lightpink",
    data_alpha=0.4,
    fig_save_dir=Path("/src/project/figures/learning/"),
    fig_name="priors.pdf",
    labels=["True prior", "Flow direct", "Flow adapted (PT)", "Flow adapted (RND)"],
    linestyles=["-", "--", "-.", ":"],
)

(1000, 45)
(1000, 45)
(1000, 45)
(1000, 45)


In [55]:
device = "cpu"
density_support = (1e-3, 2)
xticks = (0, 2)
xticklabels = (0, 2)
plot_ylim = (0, 1.5)
visualize_marginal_flow(
    models=[true_prior, direct_flow, pt_adpt_flow.prior, rnd_adpt_flow.prior],
    data_loader=val_loader,
    device=device,
    density_support=density_support,
    density_n_samples=1000,
    dims_to_plot=range(45),
    fig_dpi=300,
    linewidth=2,
    fontsize=10,
    plot_xlim=xticks,
    plot_ylim=plot_ylim,
    data_color="lightpink",
    data_alpha=0.4,
    fig_save_dir=Path("/src/project/figures/learning/"),
    fig_name="priors_head1.pdf",
    labels=["True prior", "Flow direct", "Flow adapted (PT)", "Flow adapted (RND)"],
    linestyles=["-", "--", "-.", ":"],
)

(1000, 45)
(1000, 45)
(1000, 45)
(1000, 45)


In [57]:
device = "cpu"
density_support = (1e-3, 3)
xticks = (0, 3)
xticklabels = (0, 3)
plot_ylim = (0, 1.5)
visualize_marginal_flow(
    models=[true_prior, direct_flow, pt_adpt_flow.prior, rnd_adpt_flow.prior],
    data_loader=val_loader,
    device=device,
    density_support=density_support,
    density_n_samples=1000,
    dims_to_plot=range(45),
    fig_dpi=300,
    linewidth=2,
    fontsize=10,
    plot_xlim=xticks,
    plot_ylim=plot_ylim,
    data_color="lightpink",
    data_alpha=0.4,
    fig_save_dir=Path("/src/project/figures/learning/"),
    fig_name="priors_head2.pdf",
    labels=["True prior", "Flow direct", "Flow adapted (PT)", "Flow adapted (RND)"],
    linestyles=["-", "--", "-.", ":"],
)

(1000, 45)
(1000, 45)
(1000, 45)
(1000, 45)


In [43]:
device = "cpu"
density_support = (5, 10)
xticks = (5, 10)
xticklabels = (5, 10)
plot_ylim = (0, .01)
visualize_marginal_flow(
    models=[true_prior, direct_flow, pt_adpt_flow.prior, rnd_adpt_flow.prior],
    data_loader=val_loader,
    device=device,
    density_support=density_support,
    density_n_samples=1000,
    dims_to_plot=range(45),
    fig_dpi=300,
    linewidth=2,
    fontsize=10,
    plot_xlim=xticks,
    plot_ylim=plot_ylim,
    data_color="lightpink",
    data_alpha=0.4,
    fig_save_dir=Path("/src/project/figures/learning/"),
    fig_name="priors_tail1.pdf",
    labels=["True prior", "Flow direct", "Flow adapted (PT)", "Flow adapted (RND)"],
    linestyles=["-", "--", "-.", ":"],
)

(1000, 45)
(1000, 45)
(1000, 45)
(1000, 45)


In [48]:
device = "cpu"
density_support = (10, 15)
xticks = density_support
xticklabels = density_support
plot_ylim = (0, 0.0001)
visualize_marginal_flow(
    models=[true_prior, direct_flow, pt_adpt_flow.prior, rnd_adpt_flow.prior],
    data_loader=val_loader,
    device=device,
    density_support=density_support,
    density_n_samples=1000,
    dims_to_plot=range(45),
    fig_dpi=300,
    linewidth=2,
    fontsize=10,
    plot_xlim=xticks,
    plot_ylim=plot_ylim,
    data_color="lightpink",
    data_alpha=0.4,
    fig_save_dir=Path("/src/project/figures/learning/"),
    fig_name="priors_tail2.pdf",
    labels=["True prior", "Flow direct", "Flow adapted (PT)", "Flow adapted (RND)"],
    linestyles=["-", "--", "-.", ":"],
)

(1000, 45)
(1000, 45)
(1000, 45)
(1000, 45)


In [52]:
device = "cpu"
density_support = (15, 20)
xticks = density_support
xticklabels = density_support
plot_ylim = (0, 0.0001)
visualize_marginal_flow(
    models=[true_prior, direct_flow, pt_adpt_flow.prior, rnd_adpt_flow.prior],
    data_loader=val_loader,
    device=device,
    density_support=density_support,
    density_n_samples=1000,
    dims_to_plot=range(45),
    fig_dpi=300,
    linewidth=2,
    fontsize=10,
    plot_xlim=xticks,
    plot_ylim=plot_ylim,
    data_color="lightpink",
    data_alpha=0.4,
    fig_save_dir=Path("/src/project/figures/learning/"),
    fig_name="priors_tail3.pdf",
    labels=["True prior", "Flow direct", "Flow adapted (PT)", "Flow adapted (RND)"],
    linestyles=["-", "--", "-.", ":"],
)

(1000, 45)
(1000, 45)
(1000, 45)
(1000, 45)


In [77]:
def partwisehist(
    data_loader,
    device="cpu",
    dims_to_plot=range(45),
    fig_dpi=300,
    fontsize=10,
    data_color="darkorange",
    data_alpha=0.4,
    fig_save_dir=Path("/src/project/figures/learning/"),
    **catch_all,
):
    all_responses = []
    for batch in data_loader:
        responses, _ = batch
        all_responses.append(responses.to(device))
    all_responses = torch.cat(all_responses, dim=0)
    n_dims_to_plot = len(dims_to_plot)
    fig, axs = plt.subplots(
        7,
        7,
        sharey=True,
        sharex=True,
        dpi=fig_dpi,
        tight_layout=True,
    )
    for idx, ax in zip(dims_to_plot, axs.ravel()):
        sns.histplot(
            all_responses[:, idx].detach().cpu(),
            ax=ax,
            stat="probability",
            element="step",
            color=data_color,
            alpha=data_alpha,
            label="Data",
            bins=catch_all['bins'],
        )
        ax.set_xticks(catch_all['xticks'])
        ax.set_xticklabels(catch_all['xticklabels'])
        ax.set_xlabel("x", fontsize=fontsize)
        ax.set_ylabel("p(x)", fontsize=fontsize)
        ax.set_xlim(*catch_all['xlim'])
        ax.set_ylim(*catch_all['ylim'])
    for ax in axs.ravel()[n_dims_to_plot:]:
        ax.axis("off")
    handles, labels = axs.flatten()[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower right", fontsize=fontsize)
    fig.savefig(
        fig_save_dir / catch_all["fig_name"],
        bbox_inches="tight",
        transparent=True,
    )
    # close the figure to avoid memory leak
    plt.close(fig)

In [75]:
partwisehist(
    data_loader=val_loader,
    device=device,
    density_support=density_support,
    dims_to_plot=range(45),
    fig_dpi=300,
    data_color="lightpink",
    data_alpha=1,
    fig_save_dir=Path("/src/project/figures/learning/"),
    fig_name="partwisehist.pdf",
    bins=[0, 5, 10, 15],
    xticks=[0, 5, 10, 15],
    xticklabels=[0, 5, 10, 15],
)

In [79]:
partwisehist(
    data_loader=val_loader,
    device=device,
    density_support=density_support,
    dims_to_plot=range(45),
    fig_dpi=300,
    data_color="lightpink",
    data_alpha=1,
    fig_save_dir=Path("/src/project/figures/learning/"),
    fig_name="partwisehist_tail1.pdf",
    bins=[0, 5, 10, 15],
    xticks=[0, 5, 10, 15],
    xticklabels=[0, 5, 10, 15],
    xlim=(0, 15),
    ylim=(0, 0.01),
)

In [81]:
partwisehist(
    data_loader=val_loader,
    device=device,
    density_support=density_support,
    dims_to_plot=range(45),
    fig_dpi=300,
    data_color="lightpink",
    data_alpha=1,
    fig_save_dir=Path("/src/project/figures/learning/"),
    fig_name="partwisehist_tail3.pdf",
    bins=[0, 5, 10, 15, 20],
    xticks=[0, 5, 10, 15, 20],
    xticklabels=[0, 5, 10, 15, 20],
    xlim=(0, 15),
    ylim=(0, 0.001),
)